# HW5: Bigfoot Sighting Reports (BFRO)

Two Altair visualizations of the BFRO Bigfoot reports dataset, exported as Vega-Lite JSON for my Jekyll page.

Data: https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

## Read in the data (from a URL)

In [2]:
url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv'
df = pd.read_csv(url)
print(df.shape)
df[['state', 'season', 'date', 'classification', 'latitude', 'longitude']].head()

(4747, 29)


,state,season,date,classification,latitude,longitude
0,Alaska,Fall,NaN,Class A,NaN,NaN
1,New Jersey,Fall,NaN,Class B,NaN,NaN
2,New Hampshire,Summer,2016-06-07,Class A,43.41549,-72.33093
3,Nebraska,Spring,2018-05-25,Class B,42.15685,-96.34203
4,North Dakota,Spring,2000-04-21,Class A,48.25422,-101.31660


## Data transformations

- Convert `date` to a timestamp and pull out the year.
- Keep only the columns I plot (this keeps the exported JSON small).
- Drop the handful of Class C reports (only ~30 rows) so the comparison is Class A vs Class B.

In [3]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year

df_clean = df[df['classification'].isin(['Class A', 'Class B'])]
df_clean = df_clean[['state', 'season', 'year', 'classification', 'latitude', 'longitude', 'number']]
df_clean.info()

<class 'pandas.DataFrame'>
Index: 4717 entries, 0 to 4746
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   state           4717 non-null   str    
 1   season          4717 non-null   str    
 2   year            3781 non-null   float64
 3   classification  4717 non-null   str    
 4   latitude        3781 non-null   float64
 5   longitude       3781 non-null   float64
 6   number          4717 non-null   float64
dtypes: float64(4), str(3)
memory usage: 294.8 KB


## Plot 1: Reports per year by classification (interactive legend)

Aggregate in pandas to one row per year and classification, then plot as a stacked bar chart. Clicking a classification in the legend highlights it and fades the other one.

In [4]:
df_year = (df_clean.dropna(subset=['year'])
           .groupby(['year', 'classification'])
           .size()
           .reset_index(name='reports'))
df_year = df_year[df_year['year'] >= 1960]
df_year['year'] = pd.to_datetime(df_year['year'].astype(int), format='%Y')

class_select = alt.selection_point(fields=['classification'], bind='legend')

chart1 = alt.Chart(df_year).mark_bar().encode(
    x=alt.X('year:T', title='Year', timeUnit='year'),
    y=alt.Y('reports:Q', title='Number of reports'),
    color=alt.Color('classification:N', title='Classification',
                    scale=alt.Scale(domain=['Class A', 'Class B'],
                                    range=['#1b7837', '#bf812d'])),
    opacity=alt.condition(class_select, alt.value(1), alt.value(0.15)),
    tooltip=[alt.Tooltip('year:T', timeUnit='year', title='Year'),
             alt.Tooltip('classification:N', title='Classification'),
             alt.Tooltip('reports:Q', title='Reports')]
).add_params(
    class_select
).properties(
    width=650, height=320,
    title='Bigfoot reports per year, Class A vs Class B (click the legend)'
)
chart1

alt.Chart(...)

## Plot 2: Where sightings happen, filtered by season (dropdown)

Drop rows without coordinates and keep points inside the lower 48 + Alaska bounds used by the `albersUsa` projection. A dropdown picks a season; reports from that season are highlighted on the map.

In [5]:
df_map = df_clean.dropna(subset=['latitude', 'longitude'])
df_map = df_map[df_map['season'] != 'Unknown']
df_map = df_map[df_map['longitude'] < -60]   # removes a few mis-geocoded points
df_map['year'] = df_map['year'].astype('Int64')

states = alt.topo_feature('https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json', 'states')

background = alt.Chart(states).mark_geoshape(
    fill='#f0f0f0', stroke='white'
).project('albersUsa')

season_dropdown = alt.binding_select(options=['Spring', 'Summer', 'Fall', 'Winter'], name='Season: ')
season_select = alt.selection_point(fields=['season'], bind=season_dropdown, value='Summer')

points = alt.Chart(df_map).mark_circle(size=14).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    color=alt.condition(
        season_select,
        alt.Color('season:N', title='Season',
                  scale=alt.Scale(domain=['Spring', 'Summer', 'Fall', 'Winter'],
                                  range=['#66a61e', '#e6ab02', '#d95f02', '#1f78b4'])),
        alt.value('lightgray')),
    opacity=alt.condition(season_select, alt.value(0.85), alt.value(0.15)),
    tooltip=[alt.Tooltip('state:N', title='State'),
             alt.Tooltip('season:N', title='Season'),
             alt.Tooltip('year:Q', title='Year'),
             alt.Tooltip('classification:N', title='Classification')]
).add_params(
    season_select
).project('albersUsa')

chart2 = (background + points).properties(
    width=700, height=420,
    title='Where Bigfoot reports come from, by season (use the dropdown)'
)
chart2

alt.LayerChart(...)

## Export to JSON for the Jekyll page

In [6]:
chart1.save('chart1.json')
chart2.save('chart2.json')